# ПЗ 4.1.1. Диагностика узких мест при обработке больших наборов данных на одном компьютере

**Дисциплина:** Системы обработки больших данных  
**Тема:** Диагностика узких мест при обработке больших наборов данных на одном компьютере  
**Цель работы:** сравнить полную загрузку файла в память, пофайловую обработку и блочную обработку, а затем сделать вывод о том, где именно возникают узкие места: память, время, формат файла, число признаков и количество проходов по данным.

В этой работе используются учебные синтетические датасеты. Они достаточно велики, чтобы показать различия между подходами, но при этом безопасны для запуска на обычном учебном компьютере.


## 1. Что должен показать результат практической работы

После выполнения работы студент должен уметь:

1. различать полную загрузку, пофайловую и блочную обработку;
2. измерять время выполнения и приблизительное потребление памяти;
3. объяснять, почему один и тот же набор данных ведёт себя по-разному в зависимости от способа обработки;
4. делать инженерный вывод, когда допустима полная загрузка, а когда нужно переходить к потоковой или блочной схеме.


In [1]:
from pathlib import Path
import json
import gc
import time
import os
import pandas as pd
import numpy as np

try:
    import psutil
except ImportError:
    psutil = None

BASE_DIR = Path('.')
with open(BASE_DIR / 'variant_specs.json', 'r', encoding='utf-8') as f:
    VARIANTS = {item['variant']: item for item in json.load(f)}

print(f'Рабочий каталог: {BASE_DIR.resolve()}')
print(f'Доступно вариантов: {len(VARIANTS)}')

Рабочий каталог: /mnt/data/pz_4_1_1_pack
Доступно вариантов: 15


## 2. Выбор варианта

Установите номер варианта.  
Если преподаватель не назначил вариант отдельно, можно начать с **варианта По списку в журнале**.


In [2]:
VARIANT = 1
cfg = VARIANTS[VARIANT]
cfg

{'variant': 1,
 'full_file': 'sales_small.csv',
 'chunk_file': 'sales_medium.csv',
 'chunksize': 10000,
 'group_col': 'region',
 'metric': 'revenue',
 'folder': 'sensor_parts',
 'question': 'Сравнить полную загрузку sales_small.csv с блочной обработкой sales_medium.csv и пофайловой обработкой sensor_parts. Определить, на каком этапе затраты памяти и времени наиболее заметны.'}

## 3. Вспомогательные функции

В работе будем фиксировать:
- время выполнения;
- изменение занимаемой памяти процессом;
- размер файла на диске;
- простые статистики по набору данных.

> Замечание: измерение памяти в учебной работе является ориентировочным. Нам важна сравнительная картина, а не абсолютная точность до байта.


In [3]:
def rss_mb():
    if psutil is None:
        return np.nan
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)

def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2)

def summarize_frame(df, group_col=None, metric=None):
    result = {
        'rows': len(df),
        'cols': len(df.columns),
        'missing_total': int(df.isna().sum().sum())
    }
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    result['numeric_cols'] = len(numeric_cols)
    if group_col is not None and metric is not None and group_col in df.columns and metric in df.columns:
        grouped = df.groupby(group_col)[metric].agg(['count', 'mean', 'sum']).reset_index()
        result['grouped'] = grouped
    return result

def show_grouped(grouped, max_rows=10):
    if grouped is None:
        print('Групповая статистика не вычислялась.')
    else:
        display(grouped.head(max_rows))

def timed_full_read(path, group_col=None, metric=None):
    mem_before = rss_mb()
    t0 = time.perf_counter()
    df = pd.read_csv(path)
    summary = summarize_frame(df, group_col, metric)
    elapsed = time.perf_counter() - t0
    mem_after = rss_mb()
    result = {
        'method': 'full_read',
        'path': str(path),
        'file_size_mb': round(file_size_mb(path), 2),
        'elapsed_sec': round(elapsed, 4),
        'rss_before_mb': round(mem_before, 2) if not np.isnan(mem_before) else np.nan,
        'rss_after_mb': round(mem_after, 2) if not np.isnan(mem_after) else np.nan,
        'rss_delta_mb': round(mem_after - mem_before, 2) if not np.isnan(mem_before) and not np.isnan(mem_after) else np.nan,
        'rows': summary['rows'],
        'cols': summary['cols'],
        'missing_total': summary['missing_total'],
        'numeric_cols': summary['numeric_cols']
    }
    grouped = summary.get('grouped')
    return result, grouped, df

def timed_chunk_read(path, chunksize, group_col=None, metric=None):
    mem_before = rss_mb()
    t0 = time.perf_counter()
    total_rows = 0
    total_missing = 0
    cols = None
    numeric_cols = None
    aggregated = []
    for chunk in pd.read_csv(path, chunksize=chunksize):
        total_rows += len(chunk)
        total_missing += int(chunk.isna().sum().sum())
        cols = len(chunk.columns)
        numeric_cols = len(chunk.select_dtypes(include='number').columns)
        if group_col is not None and metric is not None and group_col in chunk.columns and metric in chunk.columns:
            part = chunk.groupby(group_col)[metric].agg(['count','sum']).reset_index()
            aggregated.append(part)
    elapsed = time.perf_counter() - t0
    mem_after = rss_mb()
    grouped = None
    if aggregated:
        grouped = pd.concat(aggregated, ignore_index=True).groupby(group_col)[['count','sum']].sum().reset_index()
        grouped['mean'] = grouped['sum'] / grouped['count']
        grouped = grouped[[group_col, 'count', 'mean', 'sum']]
    result = {
        'method': 'chunk_read',
        'path': str(path),
        'file_size_mb': round(file_size_mb(path), 2),
        'chunksize': chunksize,
        'elapsed_sec': round(elapsed, 4),
        'rss_before_mb': round(mem_before, 2) if not np.isnan(mem_before) else np.nan,
        'rss_after_mb': round(mem_after, 2) if not np.isnan(mem_after) else np.nan,
        'rss_delta_mb': round(mem_after - mem_before, 2) if not np.isnan(mem_before) and not np.isnan(mem_after) else np.nan,
        'rows': total_rows,
        'cols': cols,
        'missing_total': total_missing,
        'numeric_cols': numeric_cols
    }
    return result, grouped

def timed_folder_read(folder_path, group_col=None, metric=None):
    mem_before = rss_mb()
    t0 = time.perf_counter()
    total_rows = 0
    total_missing = 0
    cols = None
    numeric_cols = None
    aggregated = []
    folder = Path(folder_path)
    files = sorted(folder.glob('*.csv'))
    file_table = []
    for path in files:
        df = pd.read_csv(path)
        total_rows += len(df)
        total_missing += int(df.isna().sum().sum())
        cols = len(df.columns)
        numeric_cols = len(df.select_dtypes(include='number').columns)
        file_table.append({'file': path.name, 'rows': len(df), 'size_mb': round(file_size_mb(path), 2)})
        if group_col is not None and metric is not None and group_col in df.columns and metric in df.columns:
            part = df.groupby(group_col)[metric].agg(['count','sum']).reset_index()
            aggregated.append(part)
        del df
        gc.collect()
    elapsed = time.perf_counter() - t0
    mem_after = rss_mb()
    grouped = None
    if aggregated:
        grouped = pd.concat(aggregated, ignore_index=True).groupby(group_col)[['count','sum']].sum().reset_index()
        grouped['mean'] = grouped['sum'] / grouped['count']
        grouped = grouped[[group_col, 'count', 'mean', 'sum']]
    result = {
        'method': 'file_by_file',
        'path': str(folder_path),
        'files_count': len(files),
        'elapsed_sec': round(elapsed, 4),
        'rss_before_mb': round(mem_before, 2) if not np.isnan(mem_before) else np.nan,
        'rss_after_mb': round(mem_after, 2) if not np.isnan(mem_after) else np.nan,
        'rss_delta_mb': round(mem_after - mem_before, 2) if not np.isnan(mem_before) and not np.isnan(mem_after) else np.nan,
        'rows': total_rows,
        'cols': cols,
        'missing_total': total_missing,
        'numeric_cols': numeric_cols
    }
    return result, grouped, pd.DataFrame(file_table)

def cleanup(*objects):
    for obj in objects:
        try:
            del obj
        except:
            pass
    gc.collect()
    return 'Промежуточные объекты удалены, сборщик мусора вызван.'

## 4. Краткое описание варианта и ожидаемых файлов

In [4]:
cfg

{'variant': 1,
 'full_file': 'sales_small.csv',
 'chunk_file': 'sales_medium.csv',
 'chunksize': 10000,
 'group_col': 'region',
 'metric': 'revenue',
 'folder': 'sensor_parts',
 'question': 'Сравнить полную загрузку sales_small.csv с блочной обработкой sales_medium.csv и пофайловой обработкой sensor_parts. Определить, на каком этапе затраты памяти и времени наиболее заметны.'}

In [5]:
full_path = BASE_DIR / cfg['full_file']
chunk_path = BASE_DIR / cfg['chunk_file']
folder_path = BASE_DIR / cfg['folder']

overview = pd.DataFrame([
    {'role':'Файл для полной загрузки', 'path': full_path.name, 'exists': full_path.exists(), 'size_mb': round(file_size_mb(full_path),2)},
    {'role':'Файл для чтения по чанкам', 'path': chunk_path.name, 'exists': chunk_path.exists(), 'size_mb': round(file_size_mb(chunk_path),2)},
    {'role':'Папка для пофайловой обработки', 'path': folder_path.name, 'exists': folder_path.exists(), 'size_mb': round(sum(p.stat().st_size for p in folder_path.glob('*.csv'))/(1024**2),2)},
])
overview

,role,path,exists,size_mb
0,Файл для полной загрузки,sales_small.csv,True,2.81
1,Файл для чтения по чанкам,sales_medium.csv,True,12.53
2,Папка для пофайловой обработки,sensor_parts,True,45.78


## 5. Эксперимент 1. Полная загрузка файла в память

В этом эксперименте весь файл читается целиком с помощью `pd.read_csv()`.  
Это самый простой и интуитивно понятный способ, но именно он первым начинает создавать проблемы при росте объёма данных.


In [6]:
res_full, grouped_full, df_full = timed_full_read(
    full_path,
    group_col=cfg['group_col'],
    metric=cfg['metric']
)
pd.DataFrame([res_full])

,method,path,file_size_mb,elapsed_sec,rss_before_mb,rss_after_mb,rss_delta_mb,rows,cols,missing_total,numeric_cols
0,full_read,sales_small.csv,2.81,0.0771,356.24,369.9,13.66,50000,10,0,7


In [7]:
show_grouped(grouped_full)

,region,count,mean,sum
0,Center,10049,86.556295,869804.21
1,East,10045,86.418100,868069.81
2,North,9025,85.942049,775626.99
3,South,10924,84.330499,921226.37
4,West,9957,85.514188,851464.77


### Промежуточный вывод по эксперименту 1

Опишите:
1. сколько времени заняла полная загрузка;
2. насколько выросло потребление памяти;
3. влияет ли тип данных (например, текстовые поля или большое число столбцов) на результат;
4. удобно ли этот способ использовать для ещё более крупных файлов.


In [8]:
cleanup(df_full)

'Промежуточные объекты удалены, сборщик мусора вызван.'

## 6. Эксперимент 2. Пофайловая обработка

Теперь тот же принцип статистической обработки выполняется не над одним большим файлом, а над папкой из нескольких частей.  
Такая схема типична для журналов, сенсорных данных и архивов, где информация хранится партиями.


In [9]:
res_folder, grouped_folder, file_table = timed_folder_read(
    folder_path,
    group_col=cfg['group_col'],
    metric=cfg['metric']
)
pd.DataFrame([res_folder])

,method,path,files_count,elapsed_sec,rss_before_mb,rss_after_mb,rss_delta_mb,rows,cols,missing_total,numeric_cols
0,file_by_file,sensor_parts,8,1.3795,369.9,374.9,5.0,640000,11,0,8


In [10]:
file_table

,file,rows,size_mb
0,sensor_part_01.csv,80000,5.72
1,sensor_part_02.csv,80000,5.72
2,sensor_part_03.csv,80000,5.72
3,sensor_part_04.csv,80000,5.72
4,sensor_part_05.csv,80000,5.72
5,sensor_part_06.csv,80000,5.72
6,sensor_part_07.csv,80000,5.72
7,sensor_part_08.csv,80000,5.72


In [11]:
show_grouped(grouped_folder)

Групповая статистика не вычислялась.


### Промежуточный вывод по эксперименту 2

Опишите:
1. как изменилась память по сравнению с полной загрузкой;
2. стало ли время больше или меньше;
3. какие преимущества даёт пофайловая обработка;
4. какой недостаток есть у этого подхода (например, большое число открытий файлов, накладные расходы на чтение, повторяемость операций).


## 7. Эксперимент 3. Блочная обработка (чтение по чанкам)

Теперь используем `pd.read_csv(..., chunksize=...)`.  
Этот подход особенно полезен, когда файл большой, но не хочется дробить его вручную на части.  
Ваша задача — сопоставить этот способ с предыдущими.


In [12]:
res_chunk, grouped_chunk = timed_chunk_read(
    chunk_path,
    chunksize=cfg['chunksize'],
    group_col=cfg['group_col'],
    metric=cfg['metric']
)
pd.DataFrame([res_chunk])

,method,path,file_size_mb,chunksize,elapsed_sec,rss_before_mb,rss_after_mb,rss_delta_mb,rows,cols,missing_total,numeric_cols
0,chunk_read,sales_medium.csv,12.53,10000,0.2967,374.9,385.9,11.0,220000,10,0,7


In [13]:
show_grouped(grouped_chunk)

,region,count,mean,sum
0,Center,43961,85.720129,3768342.61
1,East,44065,85.755197,3778802.77
2,North,39647,86.081120,3412858.18
3,South,48663,85.806909,4175621.62
4,West,43664,85.307184,3724852.90


### Промежуточный вывод по эксперименту 3

Опишите:
1. как выбранный `chunksize` повлиял на время;
2. удалось ли уменьшить рост потребления памяти;
3. почему блочная обработка считается компромиссом между удобством и экономией ресурсов;
4. что изменится, если `chunksize` увеличить или уменьшить в 2 раза.


## 8. Сводное сравнение результатов


In [14]:
comparison = pd.DataFrame([res_full, res_folder, res_chunk])
comparison

,method,path,file_size_mb,elapsed_sec,rss_before_mb,rss_after_mb,rss_delta_mb,rows,cols,missing_total,numeric_cols,files_count,chunksize
0,full_read,sales_small.csv,2.81,0.0771,356.24,369.9,13.66,50000,10,0,7,NaN,NaN
1,file_by_file,sensor_parts,NaN,1.3795,369.90,374.9,5.00,640000,11,0,8,8.0,NaN
2,chunk_read,sales_medium.csv,12.53,0.2967,374.90,385.9,11.00,220000,10,0,7,NaN,10000.0


In [15]:
comparison[['method','elapsed_sec','rss_delta_mb','rows','cols','numeric_cols']]

,method,elapsed_sec,rss_delta_mb,rows,cols,numeric_cols
0,full_read,0.0771,13.66,50000,10,7
1,file_by_file,1.3795,5.00,640000,11,8
2,chunk_read,0.2967,11.00,220000,10,7


## 9. Аналитическая часть

Сформулируйте развернутые ответы:

1. Какой способ оказался самым быстрым?
2. Какой способ оказался самым экономным по памяти?
3. Где проявилась зависимость от структуры данных: числа столбцов, наличия текстовых признаков, формата хранения?
4. В каком случае полная загрузка в память остаётся допустимой?
5. Когда пофайловая обработка предпочтительнее чтения большого файла?
6. Когда чтение по чанкам удобнее разбиения на отдельные файлы?
7. Какие узкие места вы диагностировали в своём варианте: память, время, формат, число проходов по данным, количество столбцов?


## 10. Дополнительный эксперимент (для претендующих на ОТЛИЧНО)

Измените `chunksize` и повторите третью часть работы.  
Например, проверьте три варианта:
- `chunksize // 2`
- `chunksize`
- `chunksize * 2`

После этого постройте таблицу и сделайте вывод, существует ли компромисс между:
- накладными расходами на частое чтение маленьких чанков;
- ростом памяти при слишком крупных чанках.


In [16]:
test_chunk_sizes = [max(1000, cfg['chunksize']//2), cfg['chunksize'], cfg['chunksize']*2]
extra_results = []
for cs in test_chunk_sizes:
    res_tmp, _ = timed_chunk_read(
        chunk_path,
        chunksize=cs,
        group_col=cfg['group_col'],
        metric=cfg['metric']
    )
    extra_results.append(res_tmp)

pd.DataFrame(extra_results)[['method','chunksize','elapsed_sec','rss_delta_mb','rows','cols']]

,method,chunksize,elapsed_sec,rss_delta_mb,rows,cols
0,chunk_read,5000,0.3901,-0.98,220000,10
1,chunk_read,10000,0.3150,9.98,220000,10
2,chunk_read,20000,0.2747,3.25,220000,10


## 11. Требования к отчёту

В отчёт необходимо включить:

1. тему, цель работы и номер варианта;
2. краткое описание трёх способов обработки данных;
3. фрагменты кода и таблицы результатов;
4. сводную таблицу сравнения;
5. подробный вывод о диагностированных узких местах;
6. ответ на вопрос, какой способ для вашего варианта является наиболее рациональным и почему.


## 12. Контрольные вопросы

1. Почему размер файла на диске и объём памяти при обработке — не одно и то же?
2. Чем полная загрузка отличается от потоковой или блочной обработки?
3. Почему большое число столбцов может заметно увеличить расход памяти даже при умеренном числе строк?
4. В каких случаях большое количество файлов само становится источником накладных расходов?
5. Почему уменьшение `chunksize` не всегда ускоряет обработку?
6. Что в контексте этой работы можно считать узким местом?
7. Почему нельзя оценивать эффективность аналитического решения только по качеству модели, игнорируя вычислительные ресурсы?


## 13. Формула итогового вывода

Для удобства можно использовать следующую схему заключения:

> В ходе работы были сопоставлены три подхода: полная загрузка, пофайловая обработка и чтение по чанкам.  
> Установлено, что для моего варианта основным ограничением является ____________.  
> Наибольший рост памяти наблюдался при ______________________.  
> Наилучшее время показал метод ______________________.  
> Наиболее рациональным для данного класса задач является ______________________, поскольку ______________________.
